In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, classification_report

In [2]:
MODEL_CKPT = "bert-base-uncased"
dataset_name = "dair-ai/emotion"

In [3]:
id2label = {0: 'sadness', 1: 'happy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
label2id = {v: k for k, v in id2label.items()}
TARGET_LABELS = list(id2label.values())

In [4]:
# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [5]:
print(f"Loading {dataset_name}...")
dataset = load_dataset(dataset_name)

Loading dair-ai/emotion...


In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [7]:
dataset["train"].select(range(5))


Dataset({
    features: ['text', 'label'],
    num_rows: 5
})

In [8]:
dataset["train"].to_pandas()["label"].value_counts()


label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64

In [9]:
from IPython.display import Markdown, display
import pandas as pd

# Take first 3 rows (or as many as you want)
df = dataset["train"].to_pandas().head(10)

# Truncate text for display
df["text"] = df["text"].apply(lambda x: x[:30] + "…" if len(x) > 30 else x)

# Build Markdown table
md_table = df.to_markdown(index=False)

display(Markdown(md_table))


| text                            |   label |
|:--------------------------------|--------:|
| i didnt feel humiliated         |       0 |
| i can go from feeling so hopel… |       0 |
| im grabbing a minute to post i… |       3 |
| i am ever feeling nostalgic ab… |       2 |
| i am feeling grouchy            |       3 |
| ive been feeling a little burd… |       0 |
| ive been taking or milligrams … |       5 |
| i feel as confused about life … |       4 |
| i have been with petronas for … |       1 |
| i feel romantic too             |       2 |

In [10]:
# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def preprocess_function(examples):
    # Max length 128 is usually sufficient for tweets
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Tokenizing dataset...")
encoded_dataset = dataset.map(preprocess_function, batched=True)

Tokenizing dataset...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
print(f"\nLoading {MODEL_CKPT} and applying LoRA...")

# 1. Load Base Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(TARGET_LABELS),
    id2label=id2label,
    label2id=label2id
)

# 2. Define LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence Classification
    r=16,                       # Rank (8 or 16 are common)
    lora_alpha=32,              # Alpha (usually 2x rank)
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"] # Apply to BERT attention layers
)


Loading bert-base-uncased and applying LoRA...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# 3. Wrap Model with PEFT
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 594,438 || all params: 110,081,292 || trainable%: 0.5400


In [13]:
args = TrainingArguments(
    output_dir="./bert-base-lora-dair-emotion",
    eval_strategy="epoch",      # Evaluate every epoch
    save_strategy="epoch",      # Save checkpoint every epoch
    learning_rate=2e-4,         # Higher LR is often better for LoRA
    per_device_train_batch_size=32, 
    per_device_eval_batch_size=32,
    num_train_epochs=5,         # 5 epochs is usually enough for this dataset
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU available
    logging_dir='./logs',
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [15]:
from transformers import Trainer

# 1. Define a Custom Trainer to handle the compatibility issue
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # The default Trainer adds 'num_items_in_batch' to inputs, which causes the error.
        # We override this to call the model directly without that argument.
        outputs = model(**inputs)
        
        # Extract the loss
        loss = outputs.get("loss") if isinstance(outputs, dict) else outputs[0]
        
        return (loss, outputs) if return_outputs else loss

In [16]:
trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [17]:
# 3. Start Training
print("\nStarting Training with CustomTrainer...")
trainer.train()


Starting Training with CustomTrainer...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.997429,0.499372,0.825500
2,0.399886,0.248484,0.917500
3,0.248894,0.214874,0.931000
4,0.202569,0.190228,0.931500
5,0.176231,0.180340,0.932500


TrainOutput(global_step=2500, training_loss=0.40500159606933595, metrics={'train_runtime': 281.6841, 'train_samples_per_second': 284.006, 'train_steps_per_second': 8.875, 'total_flos': 2157062893443072.0, 'train_loss': 0.40500159606933595, 'epoch': 5.0})

In [18]:
print("\nEvaluating on Test Set...")
test_results = trainer.predict(encoded_dataset["test"])


Evaluating on Test Set...


In [19]:
y_preds = np.argmax(test_results.predictions, axis=1)
y_true = test_results.label_ids

print(f"Test Accuracy: {accuracy_score(y_true, y_preds):.4f}")
print("\nClassification Report (Note: 'joy' is now labeled 'happy'):")
print(classification_report(y_true, y_preds, target_names=TARGET_LABELS))

Test Accuracy: 0.9275

Classification Report (Note: 'joy' is now labeled 'happy'):
              precision    recall  f1-score   support

     sadness       0.96      0.95      0.96       581
       happy       0.95      0.95      0.95       695
        love       0.84      0.84      0.84       159
       anger       0.92      0.94      0.93       275
        fear       0.89      0.90      0.90       224
    surprise       0.74      0.73      0.73        66

    accuracy                           0.93      2000
   macro avg       0.88      0.89      0.88      2000
weighted avg       0.93      0.93      0.93      2000



In [20]:
# Save the model
model.save_pretrained("./bert_lora_happy_model")
print("Model saved!")

Model saved!
